# MASLD-LLM C3 Evaluations Notebook

Run 1.7B evaluations on MB-85xx and MB-87xx corpora.

This notebook evaluates the fine-tuned 1.7B model on:
- MB-85xx: Progression corpus (stable/progressive binary + ordinal)
- MB-87xx: Liver→plasma corpus (protein level regression)


In [ ]:
# Cell 1: Mount Drive and setup
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Path setup
MASLD_DIR = '/content/drive/MyDrive/masld-llm'
if not os.path.exists(MASLD_DIR):
    MASLD_DIR = '/content/drive/MyDrive/masld-llm-outputs'
    os.makedirs(MASLD_DIR, exist_ok=True)

print(f'MASLD directory: {MASLD_DIR}')

In [ ]:
# Cell 2: Install dependencies
!pip install -q peft transformers accelerate datasets pandas numpy
!pip install -q torch==2.14.1

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

In [ ]:
# Cell 3: Load the 1.7B model
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = 'Qwen/Qwen3-1.7B'
adapter_path = '/content/drive/MyDrive/masld-llm/checkpoints/qwen3-1.7b-masld-sft'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map='auto'
)

# Load adapter
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

print('Model loaded successfully')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Cell 4: Evaluate MB-85xx
import gzip
import json
from pathlib import Path
import numpy as np

CORPUS_PATH = '/content/drive/MyDrive/masld-llm/prompts/MB-85xx_progression_test.jsonl.gz'
OUTPUT_PATH = '/content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld__MB-85xx.jsonl.gz'

# Load corpus
records = []
with gzip.open(CORPUS_PATH, 'rt') as f:
    for line in f:
        if line.strip().startswith('{"__meta__"'):
            continue
        records.append(json.loads(line))

print(f'Loaded {len(records)} records')
print(f'Tasks: {set(r["task"] for r in records)}')

# Setup generation config
gen_config = {
    'max_new_tokens': 512,
    'do_sample': False,
    'pad_token_id': tokenizer.eos_token_id,
}

# Evaluate
results = []
for i, rec in enumerate(records):
    prompt = rec['prompt']
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_config)
    
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract answer (after prompt)
    answer = response[len(prompt):].strip()
    
    results.append({
        **rec,
        'response': response,
        'answer': answer
    })
    
    if (i + 1) % 10 == 0:
        print(f'Processed {i + 1}/{len(records)}')

print(f'\nEvaluation complete: {len(results)} records')

# Save results
with gzip.open(OUTPUT_PATH, 'wt') as f:
    f.write(json.dumps({'__meta__': {'n_records': len(results), 'model': 'qwen3-1.7b-masld'}}, indent=1) + '\n')
    for r in results:
        f.write(json.dumps(r) + '\n')

print(f'Results saved to {OUTPUT_PATH}')

In [ ]:
# Cell 5: Evaluate MB-87xx
CORPUS_PATH = '/content/drive/MyDrive/masld-llm/prompts/MB-87xx_liverplasma_test.jsonl.gz'
OUTPUT_PATH = '/content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld__MB-87xx.jsonl.gz'

# Load corpus
records = []
with gzip.open(CORPUS_PATH, 'rt') as f:
    for line in f:
        if line.strip().startswith('{"__meta__"'):
            continue
        records.append(json.loads(line))

print(f'Loaded {len(records)} records')
print(f'Tasks: {set(r["task"] for r in records)}')

# Evaluate with same config
results = []
for i, rec in enumerate(records):
    prompt = rec['prompt']
    
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_config)
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response[len(prompt):].strip()
    
    results.append({
        **rec,
        'response': response,
        'answer': answer
    })
    
    if (i + 1) % 50 == 0:
        print(f'Processed {i + 1}/{len(records)}')

print(f'\nEvaluation complete: {len(results)} records')

# Save results
with gzip.open(OUTPUT_PATH, 'wt') as f:
    f.write(json.dumps({'__meta__': {'n_records': len(results), 'model': 'qwen3-1.7b-masld'}}, indent=1) + '\n')
    for r in results:
        f.write(json.dumps(r) + '\n')

print(f'Results saved to {OUTPUT_PATH}')

In [ ]:
# Cell 6: Verify outputs
import os

eval_files = [
    '/content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld__MB-85xx.jsonl.gz',
    '/content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld__MB-87xx.jsonl.gz'
]

for fpath in eval_files:
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        with gzip.open(fpath, 'rt') as f:
            lines = len(f.readlines())
        print(f'{os.path.basename(fpath)}: {lines} lines, {size} bytes')
    else:
        print(f'{os.path.basename(fpath)}: NOT FOUND')